# HUẤN LUYỆN PRE-TRAINING MÔ HÌNH BASE TRÊN GOOGLE COLAB PRO
## (DeepSeek MLA Architecture + FineWeb-Edu / SmolLM)

Notebook này được thiết kế để chạy trực tiếp trên GPU **A100 (40GB/80GB)** hoặc **L4 (24GB)** của Colab Pro.
Mọi checkpoint sẽ tự động được lưu trữ vào **Google Drive** để chống mất mát khi bị ngắt kết nối.

### 1. Kiểm Tra Phần Cứng GPU & Mount Google Drive

In [ ]:
import os
import torch

print(f"PyTorch Version: {torch.__version__}")
assert torch.cuda.is_available(), "Vui lòng bật GPU trong Runtime -> Change runtime type!"
print(f"GPU hiện tại: {torch.cuda.get_device_name(0)}")
print(f"Dung lượng VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

# Mount Google Drive để lưu checkpoint an toàn
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = "/content/drive/MyDrive/R&DTeam/llm_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Thư mục lưu Checkpoint: {CHECKPOINT_DIR}")

PyTorch Version: 2.11.0+cu130
GPU hiện tại: NVIDIA A100-SXM4-80GB
Dung lượng VRAM: 79.25 GB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thư mục lưu Checkpoint: /content/drive/MyDrive/R&DTeam/llm_checkpoints


### 2. Cài Đặt Thư Viện Cần Thiết

In [ ]:
!pip install datasets regex matplotlib

# Kéo mã nguồn từ GitHub hoặc copy trực tiếp mã nguồn vào Colab
# !git clone https://github.com/<your-username>/llm.git /content/llm
# %cd /content/llm

### 3. Cấu Hình Kiến Trúc Mô Hình & Siêu Tham Số Huấn Luyện (Training Config)

In [ ]:
from configs.model_config import LLMConfig, AttentionType
from core.model import ModernDecoderOnlyLLM
from core.tokenizer import ByteLevelBPETokenizer

# Nạp Tokenizer đã huấn luyện (hoặc train trên tập ngữ liệu mới)
tokenizer = ByteLevelBPETokenizer.load("data/tokenizer_tinystories.json")
print(f"Vocab Size: {tokenizer.vocab_size}")

# Cấu hình kiến trúc quy mô lớn cho Colab Pro (A100)
# Bạn có thể tùy chỉnh kích thước theo mục tiêu
config = LLMConfig(
    vocab_size=tokenizer.vocab_size,
    hidden_dim=1024,          # d_model
    num_layers=12,            # 12 decoder blocks
    num_heads=16,             # 16 attention heads
    intermediate_dim=2816,    # SwiGLU FFN
    max_seq_len=1024,         # Context length
    attention_type=AttentionType.MLA,
    kv_lora_rank=256,         # Nén Latent KV Cache
    qk_rope_head_dim=64,      # Decoupled RoPE Key
    head_dim=64,
)

device = torch.device("cuda")
model = ModernDecoderOnlyLLM(config, device=device)
total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Khởi tạo mô hình thành công: {total_params:,} tham số")

### 4. Nạp Dữ Liệu Stream (FineWeb-Edu hoặc SmolLM) & Khởi Chạy Pre-Training

In [ ]:
from datasets import load_dataset
from data.dataset import create_pretraining_dataloader
from train import train_llm

# Stream tập dữ liệu giáo dục chất lượng cao FineWeb-Edu
print("Đang kết nối luồng dữ liệu HuggingFace...")
ds = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)

# Tạo DataLoader với cơ chế Token Packing (không lãng phí padding)
dataloader = create_pretraining_dataloader(
    text_stream=ds,
    tokenizer=tokenizer,
    batch_size=16,            # Micro-batch size trên A100
    seq_len=config.max_seq_len,
)

# Khởi chạy Huấn luyện Pre-training
loss_history = train_llm(
    model=model,
    dataloader=dataloader,
    tokenizer=tokenizer,
    max_steps=5000,           # Số steps huấn luyện mục tiêu
    grad_accum_steps=4,       # Effective batch size = 64
    peak_lr=5e-4,
    min_lr=5e-5,
    warmup_steps=100,
    save_interval=500,        # Tự động lưu checkpoint vào Drive mỗi 500 steps
    eval_interval=100,        # Sinh văn bản kiểm tra chất lượng mỗi 100 steps
    checkpoint_dir=CHECKPOINT_DIR,
    device=device,
)